# 03 - Leakage-Free Dataset Splitting & Factorial Perturbation Engine
**Project:** Robustness of Roman Urdu Sentiment Classification to Orthographic Spelling Variation  
**Course:** Data Science Tools & Techniques (DS T&T)  
**Author:** Danyal Ahmed Malik (26i-8030)

---
### Pipeline Overview & Objectives
1. **Leakage-Free Partitioning:** Split cleaned corpus (`23,458` records) into Stratified Train (70%), Validation (15%), and Test (15%) partitions with fixed random seed (`42`).
2. **Canonical Severity Conditions:**
   - `Clean`: 0% perturbation.
   - `Mild`: 25% substitution rate using random variants.
   - `Moderate`: 50% substitution rate using random variants.
   - `Severe`: 90% substitution rate using random variants.
3. **$3 	imes 3$ Factorial Grid:** Disentangle word substitution rate ($\{25\%, 50\%, 90\%\}$) from character edit distance ($\{\text{Low}, \text{Random}, \text{High}\}$).
4. **Data Augmentation:** Generate `train_augmented.csv` by injecting mild-to-moderate authentic variants into 50% of the training split to train robust models.


In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


### 1. Partitioning Verification
Check the exact class distribution across Train, Validation, and Test partitions.


In [2]:
split_dist_path = os.path.join(PROJECT_ROOT, "results", "tables", "dataset_splits_distribution.csv")
split_df = pd.read_csv(split_dist_path)
display(split_df)

split_meta_path = os.path.join(PROJECT_ROOT, "results", "tables", "dataset_splits_meta.json")
with open(split_meta_path, "r", encoding="utf-8") as f:
    split_meta = json.load(f)
print(json.dumps(split_meta, indent=2))


{
  "random_state": 42,
  "total_records": 23458,
  "train_records": 16420,
  "val_records": 3519,
  "test_records": 3519,
  "class_distributions": [
    {
      "Class": "very negative",
      "Train_Count": 1809,
      "Train_Pct": 11.02,
      "Val_Count": 388,
      "Val_Pct": 11.03,
      "Test_Count": 388,
      "Test_Pct": 11.03
    },
    {
      "Class": "negative",
      "Train_Count": 4012,
      "Train_Pct": 24.43,
      "Val_Count": 859,
      "Val_Pct": 24.41,
      "Test_Count": 860,
      "Test_Pct": 24.44
    },
    {
      "Class": "neutral",
      "Train_Count": 4246,
      "Train_Pct": 25.86,
      "Val_Count": 910,
      "Val_Pct": 25.86,
      "Test_Count": 910,
      "Test_Pct": 25.86
    },
    {
      "Class": "positive",
      "Train_Count": 4142,
      "Train_Pct": 25.23,
      "Val_Count": 888,
      "Val_Pct": 25.23,
      "Test_Count": 887,
      "Test_Pct": 25.21
    },
    {
      "Class": "very positive",
      "Train_Count": 2211,
      "Train_Pct": 13

,Class,Train_Count,Train_Pct,Val_Count,Val_Pct,Test_Count,Test_Pct
0,very negative,1809,11.02,388,11.03,388,11.03
1,negative,4012,24.43,859,24.41,860,24.44
2,neutral,4246,25.86,910,25.86,910,25.86
3,positive,4142,25.23,888,25.23,887,25.21
4,very positive,2211,13.47,474,13.47,474,13.47


### 2. Perturbation Conditions & Factorial Grid
Inspect the generated test perturbation conditions.


In [3]:
cond_path = os.path.join(PROJECT_ROOT, "results", "tables", "perturbation_conditions_summary.csv")
cond_df = pd.read_csv(cond_path)
display(cond_df)


,condition,target_rate,distance_strategy,total_reviews,mean_actual_rate,mean_edit_distance,total_perturbed_tokens
0,clean,0.00,none,3519,0.000000,0.000000,0
1,mild,0.20,low,3519,0.283893,1.061125,5988
2,moderate,0.50,random,3519,0.504065,1.328475,13827
3,severe,0.85,high,3519,0.799966,1.673810,23167
4,grid_rate25_low,0.25,low,3519,0.316458,1.060283,7286
5,grid_rate25_random,0.25,random,3519,0.316458,1.324704,7286
6,grid_rate25_high,0.25,high,3519,0.316458,1.674866,7286
7,grid_rate50_low,0.50,low,3519,0.504065,1.059808,13827
8,grid_rate50_random,0.50,random,3519,0.504065,1.328475,13827
9,grid_rate50_high,0.50,high,3519,0.504065,1.676921,13827


### 3. Verification of Perturbation Effect on Sample Reviews
Inspect raw vs perturbed reviews to verify authentic substitution fidelity.


In [4]:
from src.perturbation_engine import PerturbationEngine

engine = PerturbationEngine()
sample_text = "yeh bohot achi aur khoobsurat cheez hai lekin service bilkul kharab thi"
print("Original Text:       ", sample_text)
print("Mild (25% noise):    ", engine.perturb_text(sample_text, rate=0.25, distance_strategy='random'))
print("Moderate (50% noise):", engine.perturb_text(sample_text, rate=0.50, distance_strategy='random'))
print("Severe (90% noise):  ", engine.perturb_text(sample_text, rate=0.90, distance_strategy='high'))


Original Text:        yeh bohot achi aur khoobsurat cheez hai lekin service bilkul kharab thi
Mild (25% noise):     ('yai bohot aachhee aur khoobsurat cheez hai lekin service bilkl kharab thi', {'eligible_words': 11, 'perturbed_words': 3, 'actual_rate': 0.2727272727272727, 'mean_edit_distance': 2.3333333333333335})
Moderate (50% noise): ('yai bohot achi orr khobsorat cheez hai laiken service bilkul kharrab thee', {'eligible_words': 11, 'perturbed_words': 6, 'actual_rate': 0.5454545454545454, 'mean_edit_distance': 2.0})
Severe (90% noise):   ('yai bahhot aachhee orr khubsoorat chezzz hy laiken service blkl kharab thee', {'eligible_words': 11, 'perturbed_words': 10, 'actual_rate': 0.9090909090909091, 'mean_edit_distance': 2.5})
